In [1]:
import os
import re
import json
import unicodedata
from collections import Counter
from typing import Dict, List, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm

# Fallback para evitar problemas com versões diferentes
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

try:
    from langchain_ollama import OllamaEmbeddings, ChatOllama
except ImportError:
    from langchain_community.embeddings import OllamaEmbeddings
    from langchain_community.chat_models import ChatOllama

from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import os

print("Pasta atual:")
print(os.getcwd())

print("\nArquivos nessa pasta:")
for f in os.listdir():
    print(f)

# ============================================================
# CONFIG
# ============================================================

CROSSWALK_FILE = "SNOMED_to_CID_2.csv"
CID_BASE_FILE = "Base CID_Classificação CR.csv"
INPUT_FILE = "Resultado_SNOMED_CID_Parcial - Copia.csv"

OUTPUT_FILE = "Resultado_SNOMED_CID_Parcial_predito.csv"
CHECKPOINT_FILE = "checkpoint_classificacao_cid.csv"
PERSIST_DIR = "./chroma_cid_onco"

EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "gemma3:12b"

USE_LLM = True
BATCH_SAVE = 20
TOP_K = 12
AMBIGUITY_MARGIN = 8.0  # se top1 - top2 < isso, usa LLM


# ============================================================
# NORMALIZAÇÃO
# ============================================================

CID_RE = re.compile(r"\b([CD]\d{2}(?:\.\d{1,2})?)\b", re.I)


def normalize_text(value) -> str:
    if pd.isna(value) or value is None:
        return ""
    text = str(value).strip().lower()
    text = text.replace("_", " ")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\-\s/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_cid(value) -> str:
    if pd.isna(value) or value is None:
        return ""
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def normalize_snomed(value) -> str:
    if pd.isna(value) or value is None:
        return ""
    s = str(value).upper().strip().replace(" ", "")

    # aceita formatos: 85003, M-85003, M-8500/3
    if re.fullmatch(r"\d{5}", s):
        return f"M-{s}"

    m = re.fullmatch(r"M-(\d{4})/(\d)", s)
    if m:
        return f"M-{m.group(1)}{m.group(2)}"

    return s


def extract_cids(text: str) -> set[str]:
    if not text:
        return set()
    return {normalize_cid(m.group(1)) for m in CID_RE.finditer(str(text))}


def parse_snomed(snomed: str) -> Tuple[str, str]:
    s = normalize_snomed(snomed)

    # M-85003
    m = re.fullmatch(r"M-(\d{4})(\d)", s)
    if m:
        return m.group(1), m.group(2)

    return "", ""


def behavior_hint_from_snomed(snomed: str) -> str:
    _, behavior = parse_snomed(snomed)

    hints = {
        "0": "SNOMED indica comportamento benigno. Priorizar D10-D36.",
        "1": "SNOMED indica comportamento incerto. Priorizar D37-D48.",
        "2": "SNOMED indica in situ. Priorizar D00-D09.",
        "3": "SNOMED indica neoplasia maligna primária. Priorizar C00-C97.",
        "6": "SNOMED indica metástase. Avaliar C77-C79 quando compatível."
    }
    return hints.get(behavior, "")


def text_tokens(text: str) -> set[str]:
    return {tok for tok in normalize_text(text).split() if len(tok) >= 3}


def read_csv_flex(path: str) -> pd.DataFrame:
    for sep in [";", ","]:
        try:
            return pd.read_csv(path, sep=sep, dtype=str).fillna("")
        except Exception:
            continue
    raise ValueError(f"Não foi possível ler o arquivo: {path}")


# ============================================================
# ENGINE
# ============================================================

class EngineCIDOncologico:
    def __init__(
        self,
        crosswalk_file: str,
        cid_base_file: Optional[str] = None,
        persist_dir: str = PERSIST_DIR,
        embed_model: str = EMBED_MODEL,
        llm_model: str = LLM_MODEL,
        use_llm: bool = USE_LLM,
    ):
        self.crosswalk_file = crosswalk_file
        self.cid_base_file = cid_base_file
        self.persist_dir = persist_dir
        self.embed_model = embed_model
        self.llm_model = llm_model
        self.use_llm = use_llm

        self.crosswalk_df = pd.DataFrame()
        self.cid_base_df = pd.DataFrame()
        self.cid_lookup = {}
        self.exact_snomed_topo = {}
        self.exact_diag_topo = {}
        self.snomed_majority = {}
        self.vectorstore = None
        self.llm = None

        self._load_data()
        self._build_indexes()
        self._build_vectorstore()
        if self.use_llm:
            self._build_llm()

    def _load_data(self):
        self.crosswalk_df = read_csv_flex(self.crosswalk_file).copy()
        self.crosswalk_df = self.crosswalk_df.rename(columns={
            "Diagnóstico": "diagnostico",
            "Topografia": "topografia",
            "CID": "cid",
            "Descrição CID": "descricao_cid",
            "Classificação Final (2025)": "classificacao_final",
            "SNOMED": "snomed"
        })

        self.crosswalk_df["diagnostico_norm"] = self.crosswalk_df["diagnostico"].map(normalize_text)
        self.crosswalk_df["topografia_norm"] = self.crosswalk_df["topografia"].map(normalize_text)
        self.crosswalk_df["snomed_norm"] = self.crosswalk_df["snomed"].map(normalize_snomed)
        self.crosswalk_df["cid_norm"] = self.crosswalk_df["cid"].map(normalize_cid)

        parsed = self.crosswalk_df["snomed_norm"].map(parse_snomed)
        self.crosswalk_df["morph_code"] = parsed.map(lambda x: x[0])
        self.crosswalk_df["behavior_code"] = parsed.map(lambda x: x[1])

        self.crosswalk_df = self.crosswalk_df[self.crosswalk_df["cid_norm"] != ""].copy()

        if self.cid_base_file and os.path.exists(self.cid_base_file):
            self.cid_base_df = read_csv_flex(self.cid_base_file).copy()
            self.cid_base_df = self.cid_base_df.rename(columns={
                "Descrição CID": "descricao_cid",
                "Classificação Final (2025)": "classificacao_final",
                "CID": "cid"
            })
            self.cid_base_df["cid_norm"] = self.cid_base_df["cid"].map(normalize_cid)

    def _build_indexes(self):
        # dicionário CID -> descrição/classificação
        cid_from_crosswalk = (
            self.crosswalk_df[["cid_norm", "descricao_cid", "classificacao_final"]]
            .drop_duplicates(subset=["cid_norm"])
        )

        if not self.cid_base_df.empty:
            cid_from_base = (
                self.cid_base_df[["cid_norm", "descricao_cid", "classificacao_final"]]
                .drop_duplicates(subset=["cid_norm"])
            )
            cid_all = pd.concat([cid_from_base, cid_from_crosswalk], ignore_index=True)
        else:
            cid_all = cid_from_crosswalk.copy()

        cid_all = cid_all.drop_duplicates(subset=["cid_norm"])
        self.cid_lookup = cid_all.set_index("cid_norm").to_dict(orient="index")

        # match exato SNOMED + topografia
        self.exact_snomed_topo = (
            self.crosswalk_df[
                (self.crosswalk_df["snomed_norm"] != "") &
                (self.crosswalk_df["topografia_norm"] != "")
            ]
            .groupby(["snomed_norm", "topografia_norm"])["cid_norm"]
            .agg(lambda s: Counter(s).most_common(1)[0][0])
            .to_dict()
        )

        # match exato diagnóstico + topografia
        self.exact_diag_topo = (
            self.crosswalk_df[
                (self.crosswalk_df["diagnostico_norm"] != "") &
                (self.crosswalk_df["topografia_norm"] != "")
            ]
            .groupby(["diagnostico_norm", "topografia_norm"])["cid_norm"]
            .agg(lambda s: Counter(s).most_common(1)[0][0])
            .to_dict()
        )

        # maioria por SNOMED
        self.snomed_majority = (
            self.crosswalk_df[self.crosswalk_df["snomed_norm"] != ""]
            .groupby("snomed_norm")["cid_norm"]
            .agg(lambda s: Counter(s).most_common())
            .to_dict()
        )

    def _build_vectorstore(self):
        documents = []

        for idx, row in self.crosswalk_df.iterrows():
            content = (
                f"SNOMED: {row['snomed']}. "
                f"Diagnóstico: {row['diagnostico']}. "
                f"Topografia: {row['topografia']}. "
                f"CID correspondente: {row['cid_norm']}. "
                f"Descrição CID: {row.get('descricao_cid', '')}. "
                f"Classificação final: {row.get('classificacao_final', '')}."
            )

            documents.append(Document(
                page_content=content,
                metadata={
                    "row_id": int(idx),
                    "snomed_norm": row["snomed_norm"],
                    "diagnostico_norm": row["diagnostico_norm"],
                    "topografia_norm": row["topografia_norm"],
                    "morph_code": row["morph_code"],
                    "behavior_code": row["behavior_code"],
                    "cid_norm": row["cid_norm"],
                    "descricao_cid": row.get("descricao_cid", ""),
                    "classificacao_final": row.get("classificacao_final", "")
                }
            ))

        embeddings = OllamaEmbeddings(model=self.embed_model)

        if os.path.exists(self.persist_dir) and os.listdir(self.persist_dir):
            self.vectorstore = Chroma(
                persist_directory=self.persist_dir,
                embedding_function=embeddings
            )
        else:
            self.vectorstore = Chroma.from_documents(
                documents=documents,
                embedding=embeddings,
                persist_directory=self.persist_dir
            )

    def _build_llm(self):
        self.llm = ChatOllama(
            model=self.llm_model,
            temperature=0,
            num_ctx=4096
        )

    def _score_candidate(self, diagnosis: str, topography: str, snomed: str, meta: Dict, distance: Optional[float] = None) -> float:
        diag_norm = normalize_text(diagnosis)
        topo_norm = normalize_text(topography)
        snomed_norm = normalize_snomed(snomed)

        query_morph, query_behavior = parse_snomed(snomed_norm)

        score = 0.0

        if snomed_norm and meta.get("snomed_norm") == snomed_norm:
            score += 50

        if query_morph and meta.get("morph_code") == query_morph:
            score += 18

        if query_behavior and meta.get("behavior_code") == query_behavior:
            score += 8

        cand_topo = meta.get("topografia_norm", "")
        if topo_norm and cand_topo == topo_norm:
            score += 22
        else:
            q_top = text_tokens(topo_norm)
            c_top = text_tokens(cand_topo)
            if q_top and c_top:
                score += (len(q_top & c_top) / max(len(q_top), 1)) * 14

        cand_diag = meta.get("diagnostico_norm", "")
        if diag_norm and cand_diag == diag_norm:
            score += 20
        else:
            q_diag = text_tokens(diag_norm)
            c_diag = text_tokens(cand_diag)
            if q_diag and c_diag:
                score += (len(q_diag & c_diag) / max(len(q_diag), 1)) * 16

        if distance is not None:
            score += max(0, 10 - float(distance))

        return round(score, 4)

    def retrieve_candidates(self, diagnosis: str, topography: str, snomed: str, k: int = TOP_K) -> List[Dict]:
        query = (
            f"Classificar CID oncológico. "
            f"SNOMED: {snomed}. "
            f"Diagnóstico: {diagnosis}. "
            f"Topografia: {topography}."
        )

        hits = self.vectorstore.similarity_search_with_score(query, k=k)

        candidates = []
        seen = set()

        for doc, distance in hits:
            meta = doc.metadata
            cid = meta.get("cid_norm", "")
            if not cid or cid in seen:
                continue

            score = self._score_candidate(
                diagnosis=diagnosis,
                topography=topography,
                snomed=snomed,
                meta=meta,
                distance=distance
            )

            candidates.append({
                "cid": cid,
                "score": score,
                "distance": float(distance),
                "descricao_cid": meta.get("descricao_cid", ""),
                "classificacao_final": meta.get("classificacao_final", ""),
                "snomed_norm": meta.get("snomed_norm", ""),
                "topografia_norm": meta.get("topografia_norm", ""),
                "diagnostico_norm": meta.get("diagnostico_norm", "")
            })
            seen.add(cid)

        return sorted(candidates, key=lambda x: x["score"], reverse=True)

    def _safe_json_loads(self, text: str) -> Optional[dict]:
        try:
            return json.loads(text)
        except Exception:
            pass

        m = re.search(r"\{.*\}", text, flags=re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
        return None

    def llm_choose_candidate(self, diagnosis: str, topography: str, snomed: str, candidates: List[Dict]) -> Optional[dict]:
        if not self.llm or not candidates:
            return None

        prompt = ChatPromptTemplate.from_template("""
Você é um auditor médico especialista em codificação oncológica.

Escolha SOMENTE entre os candidatos abaixo.

CASO
SNOMED: {snomed}
Diagnóstico: {diagnosis}
Topografia: {topography}

DICA DE COMPORTAMENTO
{behavior_hint}

CANDIDATOS
{candidates_json}

REGRAS
1. Priorize SNOMED exato.
2. Depois priorize morfologia SNOMED + topografia.
3. Responda SOMENTE em JSON válido.
4. Escolha apenas um CID presente na lista.
5. Se nenhum for aceitável, responda INDETERMINADO.

Formato:
{{
  "cid": "C50",
  "reason": "motivo curto",
  "confidence": 0.0
}}
""")

        chain = prompt | self.llm | StrOutputParser()

        raw = chain.invoke({
            "snomed": snomed,
            "diagnosis": diagnosis,
            "topography": topography,
            "behavior_hint": behavior_hint_from_snomed(snomed),
            "candidates_json": json.dumps(candidates[:6], ensure_ascii=False, indent=2)
        })

        return self._safe_json_loads(raw)

    def classify_case(self, diagnosis: str, topography: str, snomed: str) -> Dict:
        diag_norm = normalize_text(diagnosis)
        topo_norm = normalize_text(topography)
        snomed_norm = normalize_snomed(snomed)

        # 1) match exato SNOMED + topografia
        cid = self.exact_snomed_topo.get((snomed_norm, topo_norm))
        if cid:
            info = self.cid_lookup.get(cid, {})
            return {
                "CID_Final": cid,
                "Metodo": "REGRA_EXATA_SNOMED_TOPO",
                "Confidence": 0.99,
                "Descricao_CID": info.get("descricao_cid", ""),
                "Classificacao_Final": info.get("classificacao_final", ""),
                "Motivo": "Casamento exato entre SNOMED e topografia."
            }

        # 2) match exato diagnóstico + topografia
        cid = self.exact_diag_topo.get((diag_norm, topo_norm))
        if cid:
            info = self.cid_lookup.get(cid, {})
            return {
                "CID_Final": cid,
                "Metodo": "REGRA_EXATA_DIAG_TOPO",
                "Confidence": 0.97,
                "Descricao_CID": info.get("descricao_cid", ""),
                "Classificacao_Final": info.get("classificacao_final", ""),
                "Motivo": "Casamento exato entre diagnóstico e topografia."
            }

        # 3) maioria por SNOMED
        if snomed_norm in self.snomed_majority:
            counts = self.snomed_majority[snomed_norm]
            total = sum(v for _, v in counts)
            best_cid, best_n = counts[0]
            dominance = best_n / total if total else 0.0

            if dominance >= 0.90:
                info = self.cid_lookup.get(best_cid, {})
                return {
                    "CID_Final": best_cid,
                    "Metodo": "REGRA_SNOMED_MAJORITARIO",
                    "Confidence": round(dominance, 4),
                    "Descricao_CID": info.get("descricao_cid", ""),
                    "Classificacao_Final": info.get("classificacao_final", ""),
                    "Motivo": f"SNOMED com CID majoritário claro ({best_n}/{total})."
                }

        # 4) recuperação vetorial + rerank
        candidates = self.retrieve_candidates(diagnosis, topography, snomed, k=TOP_K)

        if not candidates:
            return {
                "CID_Final": "INDETERMINADO",
                "Metodo": "SEM_CANDIDATOS",
                "Confidence": 0.0,
                "Descricao_CID": "",
                "Classificacao_Final": "",
                "Motivo": "Nenhum candidato recuperado."
            }

        # 5) decisão sem LLM quando top1 é claramente superior
        if len(candidates) == 1 or (len(candidates) > 1 and (candidates[0]["score"] - candidates[1]["score"] >= AMBIGUITY_MARGIN)) or not self.use_llm:
            best = candidates[0]
            info = self.cid_lookup.get(best["cid"], {})
            return {
                "CID_Final": best["cid"],
                "Metodo": "RAG_RERANK",
                "Confidence": round(min(0.95, max(0.20, best["score"] / 100)), 4),
                "Descricao_CID": info.get("descricao_cid", best.get("descricao_cid", "")),
                "Classificacao_Final": info.get("classificacao_final", best.get("classificacao_final", "")),
                "Motivo": "Melhor candidato pelo reranqueamento determinístico.",
                "Top_Candidatos": json.dumps(candidates[:5], ensure_ascii=False)
            }

        # 6) LLM só para ambiguidade
        llm_ans = self.llm_choose_candidate(diagnosis, topography, snomed, candidates)
        if llm_ans:
            chosen = normalize_cid(llm_ans.get("cid", ""))
            if any(c["cid"] == chosen for c in candidates):
                info = self.cid_lookup.get(chosen, {})
                return {
                    "CID_Final": chosen,
                    "Metodo": "RAG_LLM",
                    "Confidence": float(llm_ans.get("confidence", 0.7)),
                    "Descricao_CID": info.get("descricao_cid", ""),
                    "Classificacao_Final": info.get("classificacao_final", ""),
                    "Motivo": llm_ans.get("reason", ""),
                    "Top_Candidatos": json.dumps(candidates[:5], ensure_ascii=False)
                }

        # 7) fallback final
        best = candidates[0]
        info = self.cid_lookup.get(best["cid"], {})
        return {
            "CID_Final": best["cid"],
            "Metodo": "RAG_FALLBACK_TOP1",
            "Confidence": round(min(0.90, max(0.20, best["score"] / 100)), 4),
            "Descricao_CID": info.get("descricao_cid", best.get("descricao_cid", "")),
            "Classificacao_Final": info.get("classificacao_final", best.get("classificacao_final", "")),
            "Motivo": "LLM não confirmou; usado melhor candidato do rerank.",
            "Top_Candidatos": json.dumps(candidates[:5], ensure_ascii=False)
        }


# ============================================================
# PROCESSAMENTO EM LOTE
# ============================================================

def process_file(
    engine: EngineCIDOncologico,
    input_file: str,
    output_file: str = OUTPUT_FILE,
    checkpoint_file: str = CHECKPOINT_FILE,
    batch_save: int = BATCH_SAVE
):
    df_original = read_csv_flex(input_file).copy()

    if "DesProcedimento" not in df_original.columns:
        df_original["DesProcedimento"] = ""

    # chave única
    df_original["chave_unica"] = (
        df_original.get("Diagnóstico", "").astype(str) + "|" +
        df_original.get("Topografia", "").astype(str) + "|" +
        df_original.get("SNOMED", "").astype(str)
    )

    # deduplica antes do processamento
    df_unicos = df_original.drop_duplicates(subset=["chave_unica"]).copy()
    print(f"Total de casos únicos: {len(df_unicos)}")

    resultados_processados = []

    if os.path.exists(checkpoint_file):
        try:
            df_check = read_csv_flex(checkpoint_file)
            resultados_processados = df_check.to_dict("records")
            feitas = set(df_check["chave_unica"].astype(str))
            df_pendentes = df_unicos[~df_unicos["chave_unica"].isin(feitas)].copy()
            print(f"Retomando checkpoint: {len(feitas)} já processados, {len(df_pendentes)} pendentes.")
        except Exception as e:
            print(f"Falha ao ler checkpoint, recomeçando: {e}")
            df_pendentes = df_unicos.copy()
    else:
        df_pendentes = df_unicos.copy()

    rows = df_pendentes.to_dict("records")

    for i, row in enumerate(tqdm(rows, total=len(rows), desc="Classificando casos", unit="caso"), start=1):
        diag = str(row.get("Diagnóstico", ""))
        topo = str(row.get("Topografia", ""))
        snomed = str(row.get("SNOMED", ""))

        result = engine.classify_case(diag, topo, snomed)
        result["chave_unica"] = row["chave_unica"]
        result["Diagnóstico"] = diag
        result["Topografia"] = topo
        result["SNOMED"] = snomed

        resultados_processados.append(result)

        if len(resultados_processados) % batch_save == 0:
            pd.DataFrame(resultados_processados).to_csv(
                checkpoint_file,
                sep=";",
                index=False,
                encoding="utf-8-sig"
            )

    df_resultados_unicos = pd.DataFrame(resultados_processados).drop_duplicates(subset=["chave_unica"], keep="last")

    # reaplica nos duplicados do original
    df_final = df_original.merge(
        df_resultados_unicos[
            [
                "chave_unica",
                "CID_Final",
                "Metodo",
                "Confidence",
                "Descricao_CID",
                "Classificacao_Final",
                "Motivo"
            ]
        ],
        on="chave_unica",
        how="left"
    )

    df_final.to_csv(output_file, sep=";", index=False, encoding="utf-8-sig")
    print(f"Arquivo final salvo em: {os.path.abspath(output_file)}")
    print(f"Checkpoint salvo em: {os.path.abspath(checkpoint_file)}")

    return df_final


# ============================================================
# EXECUÇÃO
# ============================================================

if __name__ == "__main__":
    engine = EngineCIDOncologico(
        crosswalk_file=CROSSWALK_FILE,
        cid_base_file=CID_BASE_FILE,
        persist_dir=PERSIST_DIR,
        embed_model=EMBED_MODEL,
        llm_model=LLM_MODEL,
        use_llm=True
    )

    df_final = process_file(
        engine=engine,
        input_file=INPUT_FILE,
        output_file=OUTPUT_FILE,
        checkpoint_file=CHECKPOINT_FILE,
        batch_save=BATCH_SAVE
    )

    print(df_final[[
        "Diagnóstico",
        "Topografia",
        "SNOMED",
        "CID_Final",
        "Metodo",
        "Confidence"
    ]].head())

Pasta atual:
c:\Users\rafae\OneDrive\4_Arquivos\Workspace\Python\IA\snomed-cid

Arquivos nessa pasta:
Base CID_Classificação CR.csv
Base CID_Classificação CR.xlsx
chroma_cid_onco
chroma_db_local
CID_paciente_CLASSIFICADO_FINAL.csv
classificacao_final.csv
diagnosticos_topografia.txt
RAG_Sistema_CID - bk copy.ipynb
RAG_Sistema_CID - bk.ipynb
RAG_Sistema_CID_Teste 2.ipynb
RAG_Sistema_CID_Teste 3.ipynb
Resultado_SNOMED_CID_Parcial - Copia.csv
Resultado_SNOMED_CID_Parcial.csv
Resultado_SNOMED_CID_Parcial_predito.csv
SNOMED_to_CID_2.csv


C:\Users\rafae\AppData\Local\Temp\ipykernel_11664\2819198234.py:289: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vectorstore = Chroma(


Total de casos únicos: 55


Classificando casos:   0%|          | 0/55 [00:00<?, ?caso/s]

Arquivo final salvo em: c:\Users\rafae\OneDrive\4_Arquivos\Workspace\Python\IA\snomed-cid\Resultado_SNOMED_CID_Parcial_predito.csv
Checkpoint salvo em: c:\Users\rafae\OneDrive\4_Arquivos\Workspace\Python\IA\snomed-cid\checkpoint_classificacao_cid.csv
                   Diagnóstico                          Topografia   SNOMED  \
0  Alteração Fibroadenomatoide  Mama esquerda - 01 Hora(Mamotomia)  M-90100   
1                     Gastrite                      Antro gástrico  M-40000   
2                        Bócio                            Tireoide  M-71600   
3           ADENOMA SERRILHADO                    Cólon ascendente  M-82110   
4           ADENOMA SERRILHADO            Pólipo de cólon sigmoide  M-82110   

       CID_Final                    Metodo  Confidence  
0  INDETERMINADO            SEM_CANDIDATOS         0.0  
1  INDETERMINADO            SEM_CANDIDATOS         0.0  
2  INDETERMINADO            SEM_CANDIDATOS         0.0  
3           D051  REGRA_SNOMED_MAJORITARIO    